In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata
import pickle

# Individual stock returns from CRSP for all firms listed in the NYSE, AMEX, and NASDAQ

In [ ]:
stock_returns = pd.read_csv('data/raw/kjtt6tbmn0gz9chy.csv')

In [ ]:
stock_returns.head()

In [ ]:
# Returns data downloaded from CRSP contains “C” and “B”, first replace them with nan.
stock_returns['RET'].replace(['B', 'C'], [np.nan, np.nan], inplace=True)
stock_returns['RET'] = stock_returns['RET'].astype(float)

# stock-level predictive characteristics
Include 94 characteristics and industry dummies

Download: https://dachxiu.chicagobooth.edu/

In [ ]:
stock_characteristics = pd.read_csv('data/raw/datashare.csv')

In [ ]:
stock_characteristics.head()

# Stock Characteristics Processing
1、Merge characteristics and returns data

2、We cross-sectionally rank all stock characteristics period-by-period and map these ranks into the [-1,1]

3、Another issue is missing characteristics, which we replace with the cross-sectional median at each month for each stock, respectively.

In [ ]:
#1、合并特征和收益数据

#2、逐期对所有股票特征进行横截面排序，并将这些排序映射到 [-1,1] 中。

#3、另一个问题是特征缺失，分别用每只股票每个月的横截面中位数来代替。

In [ ]:
# merge data
characteristics_with_return = pd.merge(stock_characteristics, stock_returns, \
                                       left_on=['permno', 'DATE'], right_on=['PERMNO', 'date'], how='left')

In [ ]:
# Delete the row where the return is nan
characteristics_with_return = characteristics_with_return.dropna(subset=['RET'])
characteristics_with_return = characteristics_with_return.drop(columns=['PERMNO', 'date'])

# Here is an example that shows that the null value is not involved in sorting and maps the feature to [-1, 1].

In [ ]:
test = np.array([[1,9,3], [4, 1, 6], [7, np.nan, 9]])
test

In [ ]:
test = rankdata(test, axis=0, nan_policy='omit') 
test

In [ ]:
test = ((test-1) / (np.nanmax(test, axis=0) - 1) - 0.5) * 2
test

In [ ]:
def Normal(x):
    
    x = rankdata(x, axis=0, nan_policy='omit')
    
    x = ((x-1) / (np.nanmax(x, axis=0) - 1) - 0.5) * 2
    
    return x

In [ ]:
# cross-sectionally rank
feature_set = stock_characteristics.columns[2:-1]
characteristics_with_return[feature_set] = characteristics_with_return.groupby('DATE')[feature_set].transform(lambda x: Normal(x.values))

In [ ]:
characteristics_with_return.head()

In [ ]:
### number of sic 行业分类
print("number of sic:{}".format(len(characteristics_with_return.sic2.unique())))

In [ ]:
#独热编码
def encode_industry_to_one_hot(df):
    df['sic2'] = df['sic2']
    industry_encoded = pd.get_dummies(df['sic2'], prefix='sic2').astype('float64')
    df = pd.concat([df, industry_encoded], axis=1)
    return df

In [ ]:
characteristics_with_return = encode_industry_to_one_hot(characteristics_with_return)

In [ ]:
characteristics_with_return.fillna(0, inplace=True)

In [ ]:
characteristics_with_return.head()

# Eight macroeconomic predictors

including dividend-price ratio (dp), earnings-price ratio (ep), book-to-market ratio (bm), net equity expansion (ntis), Treasury-bill rate (tbl), term spread (tms), default spread (dfy), and stock variance (svar)

In [ ]:
#八个宏观经济预测指标
#包括股息价格比 (dp)、盈利价格比 (ep)、账面市值比 (bm)、
#净股本扩张 (ntis)、国库券利率 (tbl)、期限利差 (tms)、
#违约利差 (dfy) 和股票方差 (svar)

In [ ]:
macroeconomic_predictors = pd.read_csv('data/raw/PredictorData2023.xlsx - Monthly.csv', thousands=",")

In [ ]:
macroeconomic_predictors.head()

dp: The Dividend Price Ratio (d/p) is the difference between the log of dividends and the log of prices.

ep: The Earnings Price Ratio (e/p) is the difference between the log of earnings and the log of prices.

tms: The Term Spread (tms) is the difference between the long term yield on government bonds and the Treasury-bill.

dfy: The Default Yield Spread (dfy) is the difference between BAA and AAA-rated corporate bond yields.

In [ ]:
#dp： 股息价格比（d/p）是股息对数与价格对数之差。

#ep：收益价格比（e/p）是收益对数与价格对数之差。

#tms： 期限利差（tms）是政府债券的长期收益率与国库券收益率之差。

#dfy： 违约收益差 (dfy) 是 BAA 级和 AAA 级公司债券收益率之差。

In [ ]:
macroeconomic_predictors['dp'] = np.log(macroeconomic_predictors['D12']) - np.log(macroeconomic_predictors['Index'])
macroeconomic_predictors['EP'] = np.log(macroeconomic_predictors['E12']) - np.log(macroeconomic_predictors['Index'])
macroeconomic_predictors['tms'] = macroeconomic_predictors['lty'] - macroeconomic_predictors['tbl']
macroeconomic_predictors['dfy'] = macroeconomic_predictors['BAA'] - macroeconomic_predictors['AAA']

In [ ]:
macroeconomic_predictors = macroeconomic_predictors[['yyyymm', 'dp', 'EP', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']]

In [ ]:
macroeconomic_predictors.head()

# Merge macroeconomic predictors and Stock Characteristics

In [ ]:
characteristics_with_return['DATE'] = characteristics_with_return['DATE'].astype(str).str[:6].astype(int)

characteristics_with_return = pd.merge(characteristics_with_return, macroeconomic_predictors, left_on=['DATE'], right_on=['yyyymm'], how='left')

In [ ]:
characteristics_with_return.head()

In [ ]:
# 计算超额收益
characteristics_with_return['RET'] = characteristics_with_return['RET'] - characteristics_with_return['tbl']

In [ ]:
macroeconomic_columns = ['dp', 'EP', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']
len(macroeconomic_columns)

In [ ]:
len(feature_set)

In [ ]:
# Save data containing only 94 features
with open('data/characteristics_with_return.pkl', 'wb') as handle:
    keep_columns = ['permno', 'DATE', 'RET'] + list(feature_set)
    pickle.dump(characteristics_with_return[keep_columns], handle)

In [ ]:
# Save data on 94 characteristics and industries
with open('data/characteristics_with_ind_return.pkl', 'wb') as handle:
    pickle.dump(characteristics_with_return.drop(macroeconomic_columns + ['yyyymm', 'sic2'], axis=1), handle)

In [ ]:
#添加交互项
def compute_interaction_terms(df, feature_set, macroeconomic_columns):
    feature_values = df[feature_set].values
    macroeconomic_values = df[macroeconomic_columns].values
    # Compute the interaction terms
    interaction_array = feature_values[:, :, np.newaxis] * macroeconomic_values[:, np.newaxis, :]
    # Generate new column names for the interaction terms
    interaction_columns = [f'{f1}_{f2}' for f1 in feature_set for f2 in macroeconomic_columns]
    # Reshape the interaction terms array and convert it to a DataFrame
    interaction_df = pd.DataFrame(interaction_array.reshape(df.shape[0], -1), columns=interaction_columns)
    # Concatenate the interaction terms DataFrame with the original DataFrame
    df = pd.concat([df,  interaction_df], axis=1)
    return df

In [ ]:
result_df = compute_interaction_terms(characteristics_with_return, feature_set, macroeconomic_columns)

In [ ]:
result_df.drop(columns=['yyyymm', 'sic2'] + macroeconomic_columns, inplace=True)

In [ ]:
result_df.head()

In [ ]:
# Save 94 characteristics, industries and interactions data
with open('data/characteristics_with_ind_interaction_return.pkl', 'wb') as handle:
    pickle.dump(result_df, handle)